# 14_finops_and_dag
Collects the evidence for the Final Report: (1) run time per daily run from `pipeline_execution_logs`, (2) a FinOps cost estimate with editable assumptions, (3) the most complex Gold query and its physical plan for the Spark UI / Query Profile screenshot, (4) the `OPTIMIZE` demo.

Run it **after** a few days of real pipeline runs so the averages mean something.

In [ ]:
spark.sql("USE workspace.steam")

## 1. Run time per day and layer (screenshot / table for the report)

In [ ]:
display(spark.sql("""
    SELECT run_date,
           ROUND(SUM(CASE WHEN layer = 'Raw-to-Bronze'    THEN duration_sec END), 1) AS raw_to_bronze_sec,
           ROUND(SUM(CASE WHEN layer = 'Bronze-to-Silver' THEN duration_sec END), 1) AS bronze_to_silver_sec,
           ROUND(SUM(CASE WHEN layer = 'Silver-to-Gold'   THEN duration_sec END), 1) AS silver_to_gold_sec,
           ROUND(SUM(CASE WHEN layer IN ('Raw-to-Bronze','Bronze-to-Silver','Silver-to-Gold')
                          THEN duration_sec END), 1)                                  AS total_pipeline_sec
    FROM audit_pipeline_runs
    WHERE status = 'Success'
    GROUP BY run_date ORDER BY run_date
"""))

## 2. FinOps estimate
Databricks Free Edition is not billed, so the cost below is an **estimate** of what the same run would cost on a paid workspace. Fill in the two assumptions from the Databricks pricing page for your cloud and compute type (serverless jobs compute), and write the values you used in the report.

The task time is the sum of the three Success rows per day. Cold-start time of the compute is not in these numbers, so keep a safety margin.

In [ ]:
DBU_PER_HOUR = 1.0     # ASSUMPTION: DBUs consumed per hour of this job. Replace with your measured / documented value.
USD_PER_DBU  = 0.40    # ASSUMPTION: price per DBU. Replace with the current price for your plan and cloud.
RUNS_PER_YEAR = 24 * 365   # hourly pipeline for an enterprise

avg_sec = spark.sql("""
    SELECT AVG(t) FROM (
      SELECT run_date, SUM(duration_sec) AS t
      FROM audit_pipeline_runs
      WHERE status = 'Success' AND layer IN ('Raw-to-Bronze','Bronze-to-Silver','Silver-to-Gold')
      GROUP BY run_date HAVING COUNT(DISTINCT layer) = 3)
""").first()[0]

if avg_sec is None:
    print("No day has all three layers logged yet. Run the full pipeline first.")
else:
    avg_sec = float(avg_sec)
    cost_run = avg_sec / 3600 * DBU_PER_HOUR * USD_PER_DBU
    print(f"average pipeline time per run : {avg_sec:,.1f} s")
    print(f"estimated cost per run        : ${cost_run:,.4f}")
    print(f"daily schedule, 1 year        : ${cost_run * 365:,.2f}")
    print(f"hourly schedule, 1 year       : ${cost_run * RUNS_PER_YEAR:,.2f}  ({RUNS_PER_YEAR:,} runs)")

If your workspace exposes billing data, this shows real consumption. It may fail on Free Edition, which is fine.

In [ ]:
try:
    display(spark.sql("""
        SELECT usage_date, sku_name, SUM(usage_quantity) AS dbus
        FROM system.billing.usage
        GROUP BY usage_date, sku_name ORDER BY usage_date DESC LIMIT 30
    """))
except Exception as e:
    print("system.billing.usage not available:", str(e)[:200])

## 3. Most complex Gold query: DAG and plan
This runs the publisher summary on top of the "latest row per game" window (window = shuffle by `game_key`, join with `dim_publisher` = shuffle or broadcast, `GROUP BY` = shuffle).
After it runs: on **serverless**, click the *View* link (query profile) under the cell output, or open **SQL > Query History** and the query profile. Screenshot the graph for the report and read the plan below to name the shuffles (`Exchange`), the scan filters (`PushedFilters`) and the join type.

In [ ]:
q = """
    SELECT p.publisher_name, COUNT(*) AS games, SUM(f.ccu) AS total_ccu, ROUND(AVG(f.approval_rate), 4) AS avg_approval
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY game_key ORDER BY date_key DESC) AS rn
          FROM fact_game_daily) f
    JOIN dim_publisher p ON f.publisher_key = p.publisher_key
    WHERE f.rn = 1 AND f.price_tier_key IN (1, 2, 3, 4)
    GROUP BY p.publisher_name
    ORDER BY total_ccu DESC LIMIT 15
"""
display(spark.sql(q))

In [ ]:
print(spark.sql("EXPLAIN FORMATTED " + q).first()[0])

## 4. Read optimization demo (Liquid Clustering)
The fact table was created with `CLUSTER BY (date_key, game_key)`. `OPTIMIZE` rewrites the files by those columns. Screenshot both outputs. In production run it on a schedule (for example weekly), not on every run.

In [ ]:
display(spark.sql("DESCRIBE DETAIL fact_game_daily").select("numFiles", "sizeInBytes", "clusteringColumns"))
spark.sql("OPTIMIZE fact_game_daily")
display(spark.sql("DESCRIBE DETAIL fact_game_daily").select("numFiles", "sizeInBytes", "clusteringColumns"))
display(spark.sql("DESCRIBE HISTORY fact_game_daily").select("version", "timestamp", "operation", "operationMetrics").limit(5))